# Fully Standalone Laya-1.2-speedup — Complete model weights and real size

This notebook packages the **entire inference model** from [vtava/Laya-1.2-speedup](https://huggingface.co/vtava/Laya-1.2-speedup) into one directory / Hugging Face model repo:

- Full pretrained **28-layer ModernBERT encoder** (all Q/K/V, FFNs, RoPE), optimized **64-context-token decision head**, type embeddings, scorer, encoder configuration and tokenizer.
- `model.safetensors` holds **all** inference weights, not only the small head. The unused auxiliary action classifier is deliberately excluded: this model returns typed-decision **option logits only**.
- **No dependency on the original Hugging Face model or TinyCeNN-LM at inference**. After one download of the standalone repo, offline `load_model(local_dir, local_files_only=True)` works from disk.
- Reports **exact parameter count, bytes of model weights, complete directory size, component sizes and original head-only/base files**, plus a numerical/decision parity test.
- Optionally publishes to **vtava/Laya-1.2-speedup-standalone** (a *separate private model repo* by default); requires your Hugging Face write login. It never overwrites `vtava/Laya-1.2-speedup`.

**This is lossless FP32 packaging, not new training or quantization.** The old V4.6 measured speedup (~1.203× on Tesla T4) remains a historical result; this export does not claim new accuracy or performance before testing.

Run all cells on Colab. A T4 GPU and enough free disk (~several GB) are recommended. Check the upstream model license before making a public copy.


In [ ]:
#@title 1. Install dependencies and select validation hardware
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@8ba9b41894c5f0708729ad8720c2b5b3d4eafa20"
import json, gc, importlib.util, hashlib
from pathlib import Path
from collections import OrderedDict
from contextlib import nullcontext
import torch
from huggingface_hub import HfApi, hf_hub_download, snapshot_download, notebook_login
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    build_checkpoint_model, normalize_question, build_sequence, collate_items, QTYPES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:",torch.__version__,"Hardware:",torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU")
def autocast():
    return torch.autocast("cuda",dtype=torch.float16) if DEVICE=="cuda" else nullcontext()


In [ ]:
#@title 2. Load existing V4.6 head and its original base model (EXPORT only)
HEAD_REPO="vtava/Laya-1.2-speedup"
DEST_REPO="vtava/Laya-1.2-speedup-standalone"
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
OUT=Path("/content/Laya_1p2_Full_Standalone") if Path("/content").exists() else Path.cwd()/"Laya_1p2_Full_Standalone"
DEST=OUT/"standalone_model"
(DEST/"encoder").mkdir(parents=True,exist_ok=True)
(DEST/"tokenizer").mkdir(exist_ok=True)
api=HfApi()
try:
    published_info=api.model_info(HEAD_REPO)
except Exception as exc:
    print("Authorizing access to original head repository:",repr(exc))
    notebook_login()
    published_info=api.model_info(HEAD_REPO)
head_cfg=json.loads(Path(hf_hub_download(HEAD_REPO,"config.json")).read_text())
assert head_cfg["source_model"]=="convaiinnovations/laya-typed-decisions",head_cfg
assert int(head_cfg["head_context_budget"])==64,head_cfg
assert head_cfg.get("all_gates_passed") is True,"Source report does not indicate passed gates"
SOURCE=head_cfg["source_model"]
SOURCE_REVISION=head_cfg.get("source_revision") or api.model_info(SOURCE).sha
SOURCE_DIR=Path(snapshot_download(SOURCE,revision=SOURCE_REVISION,allow_patterns=[
    "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"]))
HEAD_FILE=Path(hf_hub_download(HEAD_REPO,"head.safetensors"))
PUBLISHED_CODE=Path(hf_hub_download(HEAD_REPO,"modeling_laya_speedup.py"))
original,base_config=build_checkpoint_model(SOURCE_DIR)
original=original.to(DEVICE).eval().requires_grad_(False)
original.head.load_state_dict(load_file(str(HEAD_FILE),device="cpu"),strict=True)
tokenizer=AutoTokenizer.from_pretrained(str(SOURCE_DIR/"tokenizer"))
assert tokenizer.pad_token_id is not None
print("Base repository:",SOURCE,"revision:",SOURCE_REVISION)
print("Optimized head:",HEAD_REPO,"revision:",published_info.sha)
print("Base checkpoint bytes:",(SOURCE_DIR/"model.safetensors").stat().st_size)
print("Head-only checkpoint bytes:",HEAD_FILE.stat().st_size)


In [ ]:
#@title 3. Embed source-complete OFFLINE runtime (not a TinyCeNN dependency)
# This code is physically copied into the standalone model directory.
STANDALONE_CODE = "\"\"\"Fully self-contained Laya-1.2-speedup inference code.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n"
code_path=DEST/"modeling_laya_standalone.py"
code_path.write_text(STANDALONE_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("laya_full_standalone",str(code_path))
assert spec and spec.loader
standalone_mod=importlib.util.module_from_spec(spec)
spec.loader.exec_module(standalone_mod)
assert hasattr(standalone_mod,"LayaStandalone")
assert hasattr(standalone_mod,"load_model")
print("Standalone runtime written:",code_path,"bytes",code_path.stat().st_size)


In [ ]:
#@title 4. Compare full standalone architecture with published V4.6
pspec=importlib.util.spec_from_file_location("published_laya_head",str(PUBLISHED_CODE))
assert pspec and pspec.loader
published_mod=importlib.util.module_from_spec(pspec)
pspec.loader.exec_module(published_mod)
published_model=published_mod.MarkerAwareScores(original,budget=64).eval()
full_model=standalone_mod.LayaStandalone(budget=64,pretrained_components=original).eval()
Q={
  "steering":{"type":"choice","instructions":"Choose road direction.",
              "criteria":{"left":"road curves left","straight":"road is straight","right":"road curves right"}},
  "speed":{"type":"choice","instructions":"Choose safe longitudinal intent.",
           "criteria":{"accelerate":"gain speed","hold":"maintain speed","brake":"reduce speed"}},
}
STATES=[
  {"road_direction":"left","road_visibility":0.98,"speed":7.},
  {"road_direction":"straight","road_visibility":1.0,"speed":0.},
  {"road_direction":"right","road_visibility":0.61,"speed":38.},
  {"road_direction":"unknown","road_visibility":0.1,"speed":22.},
]
test_items=[]
for state in STATES:
    for rawq in Q.values():
        question=normalize_question(rawq)
        ids,marks=build_sequence(tokenizer,state,question,max_len=512,head_max_len=192)
        assert len(marks)==len(question["crit"])
        test_items.append(dict(ids=ids,markers=marks,qtype=QTYPES[question["t"]]))
test_batch=collate_items(test_items,int(tokenizer.pad_token_id))
test_batch={k:v.to(DEVICE) for k,v in test_batch.items()}
with torch.inference_mode(),autocast():
    published_logits,_=published_model(**test_batch)
    prepared_logits,_=full_model(**test_batch)
before_error=float((published_logits.float()-prepared_logits.float()).abs().max())
assert torch.allclose(published_logits.float(),prepared_logits.float(),atol=1e-5,rtol=1e-5),before_error
reference_logits=published_logits.detach().float().cpu().clone()
print("V4.6 published-loader parity verified; max absolute logit difference:",before_error)


In [ ]:
#@title 5. Serialize all pretrained and selected head weights into ONE checkpoint
original.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
config={
 "format":"laya-1p2-full-standalone-v1",
 "architecture":"LayaStandalone",
 "head_context_budget":64,
 "head_layers":int(base_config.get("head_layers",2)),
 "source_head_repo":HEAD_REPO,
 "source_head_revision":published_info.sha,
 "source_model":SOURCE,
 "source_model_revision":SOURCE_REVISION,
 "dtype_on_disk":"original FP32, no quantization",
 "model_scope":"option scores only; auxiliary action classifier omitted",
}
(DEST/"config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")
params=sum(p.numel() for p in full_model.parameters())
modules={k:sum(p.numel() for p in getattr(full_model,k).parameters())
         for k in ("encoder","head","type_emb","scorer")}
state=full_model.state_dict()
buffer_elems=sum(x.numel() for x in full_model.buffers())
state_bytes=sum(t.numel()*t.element_size() for t in state.values())
print("Serializing all model weights (not only the decision head)...")
cpu_state=OrderedDict()
for k,v in state.items():
    cpu_state[k]=v.detach().to("cpu").contiguous().clone()
weights=DEST/"model.safetensors"
save_file(cpu_state,str(weights),metadata={"format":"pt","source_revision":str(SOURCE_REVISION)})
del cpu_state, state
gc.collect()
print("Total model parameters:",f"{params:,}")
print("By component:",modules)
print("Serialized model.safetensors size:",f"{weights.stat().st_size:,} bytes")


In [ ]:
#@title 6. Completely unload source model and verify LOCAL-ONLY reload
del full_model, published_model, original, published_mod, pspec
gc.collect()
if DEVICE=="cuda":torch.cuda.empty_cache()
loaded_model,local_tokenizer=standalone_mod.load_model(
    DEST,device=DEVICE,local_files_only=True,compile_head=False)
assert local_tokenizer.pad_token_id==tokenizer.pad_token_id
for state in STATES:
    for rawq in Q.values():
        q=normalize_question(rawq)
        a=build_sequence(tokenizer,state,q,max_len=512,head_max_len=192)
        b=standalone_mod.build_sequence(local_tokenizer,state,q,max_len=512,head_max_len=192)
        assert a==b, "Tokenizer or standalone typed-decision encoding drift"
with torch.inference_mode(),autocast():
    local_logits,_=loaded_model(**test_batch)
local_logits=local_logits.detach().float().cpu()
max_diff=float((reference_logits-local_logits).abs().max())
same_preds=bool(torch.equal(reference_logits.argmax(-1),local_logits.argmax(-1)))
close_logits=bool(torch.allclose(reference_logits,local_logits,atol=2e-3,rtol=2e-3))
assert same_preds and close_logits,(max_diff,same_preds,close_logits)
print("Offline local-only reload parity PASSED")
print("Same predictions:",int((reference_logits.argmax(-1)==local_logits.argmax(-1)).sum()),"/",reference_logits.shape[0])
print("Max absolute logit difference:",max_diff)
print("Offline predict API:",loaded_model.predict(local_tokenizer,STATES[0],Q))


In [ ]:
#@title 7. Measure actual model bytes, parameters and per-component sizes
def directory_size(p): return sum(q.stat().st_size for q in Path(p).rglob("*") if q.is_file())
def MB(b): return round(b/1_000_000,3)
def MiB(b): return round(b/(1024**2),3)
base_bytes=(SOURCE_DIR/"model.safetensors").stat().st_size
head_bytes=HEAD_FILE.stat().st_size
weight_bytes=weights.stat().st_size
size_record={
 "model_id":DEST_REPO,
 "original_head_only_model_id":HEAD_REPO,
 "original_base_model_id":SOURCE,
 "base_revision":SOURCE_REVISION,
 "head_revision":published_info.sha,
 "total_parameters":int(params),
 "parameters_by_component":modules,
 "registered_buffer_elements":int(buffer_elems),
 "total_state_tensor_bytes":int(state_bytes),
 "model_safetensors_bytes":int(weight_bytes),
 "model_safetensors_MB":MB(weight_bytes),
 "model_safetensors_MiB":MiB(weight_bytes),
 "original_head_file_bytes":int(head_bytes),
 "original_base_file_bytes":int(base_bytes),
 "original_base_plus_head_bytes":int(base_bytes+head_bytes),
 "theoretical_FP32_param_bytes":int(params*4),
 "theoretical_FP16_param_bytes":int(params*2),
 "export_dtype":"original FP32",
 "offline_parity_logit_max_abs_diff":max_diff,
 "offline_parity_predictions_identical":same_preds,
 "offline_parity_logits_close":close_logits,
 "hardware":torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU",
 "metric_limitations":"synthetic tensor parity only; 200-decision audit was conducted on V4.6 previously",
}
def write_size_report():
    size_record["all_standalone_files_bytes"]=directory_size(DEST)
    size_record["all_standalone_files_MB"]=MB(size_record["all_standalone_files_bytes"])
    (DEST/"size_report.json").write_text(json.dumps(size_record,indent=2),encoding="utf-8")
write_size_report()
print("PARAMETERS:",f"{params:,}")
print("ACTUAL FULL CHECKPOINT:",f"{weight_bytes:,} bytes / {MB(weight_bytes)} MB / {MiB(weight_bytes)} MiB")
print("STANDALONE DIRECTORY:",f"{directory_size(DEST):,} bytes / {MB(directory_size(DEST))} MB")
print("HEAD ONLY:",f"{head_bytes:,} bytes / {MB(head_bytes)} MB")
print("BASE DEPENDENCY:",f"{base_bytes:,} bytes / {MB(base_bytes)} MB")
print("FP16 theoretical weight storage (NOT exported):",MB(params*2),"MB")


In [ ]:
#@title 8. Complete model card, dependencies, provenance and final storage report
model_card=f"""---
library_name: pytorch
base_model: {SOURCE}
tags:
- modernbert
- typed-decisions
- laya-v46
- standalone
- inference-optimization
---

# Laya-1.2-speedup-standalone

Self-contained FP32 **option-scoring model** derived from
[{HEAD_REPO}](https://huggingface.co/{HEAD_REPO}).
Includes the complete original 28-layer ModernBERT encoder, selected 64-token
decision head, type embeddings, scorer, tokenizer and configuration.
The source model's auxiliary action classifier is **not** included because
V4.6 only supports option-scoring decisions.

## Real measured size

- Model parameters: **{params:,}**
- Full weight checkpoint: **{weight_bytes:,} bytes ({MB(weight_bytes)} MB)**
- Preexisting decision-head-only file: **{head_bytes:,} bytes**
- Original upstream base checkpoint: **{base_bytes:,} bytes**
- No inference-time download of either ancestor repository.

## Usage

First download this **one** repository (once):

```python
from huggingface_hub import snapshot_download
local_dir = snapshot_download("{DEST_REPO}")
```

Then inference, entirely offline:

```python
from modeling_laya_standalone import load_model
model,tokenizer=load_model(local_dir,device="cuda",local_files_only=True)
answers=model.predict(tokenizer,
    {"road_direction":"left","speed":7.0},
    {"steering":{"type":"choice","instructions":"Choose road direction.",
                 "criteria":{"left":"left","straight":"straight","right":"right"}}})
print(answers)
```

Copy `modeling_laya_standalone.py` to your Python import path as needed.
Requires packages from `requirements.txt`; no TinyCeNN-LM or base-model
download during inference.

## Research caveats

The original V4.6 narrow Tesla T4 audit reported 1.203× median GPU speedup,
200/200 teacher agreement and 84.5% gold accuracy relative to native option
scoring. **Packaging has not repeated that held-out audit or benchmarked new
hardware**. This export tests output parity on synthetic examples only. The
repository name does not guarantee 1.2× speedup in all contexts.
Inspect the upstream model's license before redistributing weights.

Base revision: `{SOURCE_REVISION}`.
Head revision: `{published_info.sha}`.
"""
(DEST/"README.md").write_text(model_card,encoding="utf-8")
(DEST/"requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n",
    encoding="utf-8")
(DEST/"SOURCE_REVISIONS.json").write_text(json.dumps({
    "source_base":SOURCE,"base_revision":SOURCE_REVISION,
    "optimized_head":HEAD_REPO,"head_revision":published_info.sha,
    "loader_source":"github:vtavakkoli/TinyCeNN-LM/standalone/laya_1p2_speedup/modeling_laya_standalone.py",
},indent=2),encoding="utf-8")
write_size_report()
print("READY: standalone full model in",DEST)
print("ALL FILES:",f"{directory_size(DEST):,} bytes / {MB(directory_size(DEST))} MB")


In [ ]:
#@title 9. Optional Hugging Face upload (private by default, no old-repo overwrite)
if PUBLISH_TO_HF:
    notebook_login()
    uploader=HfApi()
    signed_in=uploader.whoami()["name"]
    assert signed_in=="vtava",f"Expected vtava account, got {signed_in}"
    assert DEST_REPO=="vtava/Laya-1.2-speedup-standalone",(
        "Refusing to overwrite the original head-only repository")
    model_url=f"https://huggingface.co/{DEST_REPO}"
    if uploader.repo_exists(DEST_REPO,repo_type="model"):
        existing=set(uploader.list_repo_files(DEST_REPO,repo_type="model"))
        if existing-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
            raise RuntimeError(
                "The standalone repo already contains files. "
                "Set ALLOW_UPDATE_EXISTING=True only to intentionally update it.")
        print("Resuming:",model_url)
    else:
        created=uploader.create_repo(
            DEST_REPO,repo_type="model",private=PRIVATE_REPO,exist_ok=False)
        print("Created:",str(created))
    print("Uploading full model file; a large multi-GB transfer may be required.")
    uploader.upload_large_folder(
        repo_id=DEST_REPO,repo_type="model",folder_path=str(DEST))
    expected={"model.safetensors","modeling_laya_standalone.py","config.json",
              "encoder/config.json","size_report.json","README.md","requirements.txt"}
    uploaded=set(uploader.list_repo_files(DEST_REPO,repo_type="model"))
    assert expected.issubset(uploaded),f"Missing uploaded files: {expected-uploaded}"
    print("Full standalone model uploaded:",model_url)
else:
    print("Upload disabled. Locally exported model:",DEST)
print("Measured full weights bytes:",weights.stat().st_size)
